# Model Training

Notebook này thực hiện: load features → split → integrity check → scaling.
Các bước training, threshold tuning và evaluation sẽ nối tiếp bên dưới.

## 1. Setup

Question: Notebook đang dùng input nào và quy ước chia dữ liệu ra sao?

In [10]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "features.csv"

ID_COL = "CONS_NO"
TARGET_COL = "FLAG"

RANDOM_STATE = 42
TRAIN_SIZE = 0.70
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15

print(f"Features path: {FEATURES_PATH}")
print(f"Split ratio: train={TRAIN_SIZE:.0%}, validation={VALIDATION_SIZE:.0%}, test={TEST_SIZE:.0%}")
print("Note: stratify=y is used to keep the Normal/Theft ratio similar in all splits.")

Features path: c:\Users\asus\Documents\HocTap\HK4\CS114 - Machine Learning\CS114 -  ML Project - Energy Theft Detection\data\processed\features.csv
Split ratio: train=70%, validation=15%, test=15%
Note: stratify=y is used to keep the Normal/Theft ratio similar in all splits.


## 2. Load Features

Question: `features.csv` có đủ `CONS_NO`, `FLAG` và các feature dùng cho model không?

In [11]:
if not FEATURES_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {FEATURES_PATH}. Hãy chạy preprocessing_v2.py và feature.py trước."
    )

df = pd.read_csv(FEATURES_PATH)

required_cols = {ID_COL, TARGET_COL}
missing_required_cols = required_cols - set(df.columns)
if missing_required_cols:
    raise ValueError(f"features.csv thiếu cột bắt buộc: {missing_required_cols}")

feature_cols = [col for col in df.columns if col not in [ID_COL, TARGET_COL]]

X = df[feature_cols].copy()
y = df[TARGET_COL].astype(int).copy()
ids = df[ID_COL].copy()

print(f"Dataset shape: {df.shape[0]:,} customers x {df.shape[1]:,} columns")
print(f"Number of model features: {len(feature_cols):,}")
print(f"Target distribution:")
display(y.value_counts().rename(index={0: "Normal", 1: "Theft"}).to_frame("count"))

print("Note: CONS_NO is kept only for tracking customers, not as a model feature.")

Dataset shape: 42,372 customers x 97 columns
Number of model features: 95
Target distribution:


,count
FLAG,
Normal,38757
Theft,3615


Note: CONS_NO is kept only for tracking customers, not as a model feature.


## 3. Train / Validation / Test Split

Question: Chia dữ liệu như thế nào để mỗi split vẫn giữ tỷ lệ `Normal` và `Theft` gần giống dataset gốc?

In [12]:
temp_size = VALIDATION_SIZE + TEST_SIZE
relative_test_size = TEST_SIZE / temp_size

X_train, X_temp, y_train, y_temp, ids_train, ids_temp = train_test_split(
    X,
    y,
    ids,
    test_size=temp_size,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_val, X_test, y_val, y_test, ids_val, ids_test = train_test_split(
    X_temp,
    y_temp,
    ids_temp,
    test_size=relative_test_size,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

y_train = y_train.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

ids_train = ids_train.reset_index(drop=True)
ids_val = ids_val.reset_index(drop=True)
ids_test = ids_test.reset_index(drop=True)

print("Split completed.")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}, y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")

Split completed.
X_train: (29660, 95), y_train: (29660,)
X_val:   (6356, 95), y_val:   (6356,)
X_test:  (6356, 95), y_test:  (6356,)


## 4. Split Check

Question: Sau khi split, tỷ lệ `Theft` trong train/validation/test có còn gần giống nhau không?

In [13]:
def summarize_split(name: str, y_split: pd.Series) -> dict:
    counts = y_split.value_counts().sort_index()
    normal_count = counts.get(0, 0)
    theft_count = counts.get(1, 0)
    total = len(y_split)

    return {
        "split": name,
        "total": total,
        "Normal count": normal_count,
        "Theft count": theft_count,
        "Normal ratio": normal_count / total,
        "Theft ratio": theft_count / total,
    }

split_report = pd.DataFrame(
    [
        summarize_split("full", y),
        summarize_split("train", y_train),
        summarize_split("validation", y_val),
        summarize_split("test", y_test),
    ]
)

display(
    split_report.style.format(
        {
            "Normal ratio": "{:.2%}",
            "Theft ratio": "{:.2%}",
        }
    )
)

print("Observation: Nếu các Theft ratio gần bằng nhau, stratified split đã hoạt động đúng.")

,split,total,Normal count,Theft count,Normal ratio,Theft ratio
0,full,42372,38757,3615,91.47%,8.53%
1,train,29660,27130,2530,91.47%,8.53%
2,validation,6356,5813,543,91.46%,8.54%
3,test,6356,5814,542,91.47%,8.53%


Observation: Nếu các Theft ratio gần bằng nhau, stratified split đã hoạt động đúng.


## 5. Split Integrity Check

Question: Có customer nào bị trùng lặp giữa các split không? Tổng số row có khớp không?

In [14]:
assert len(X_train) + len(X_val) + len(X_test) == len(X), (
    f"Row count mismatch: {len(X_train)} + {len(X_val)} + {len(X_test)} != {len(X)}"
)

assert ids_train.isin(ids_val).sum() == 0, "Overlap detected: train ∩ validation"
assert ids_train.isin(ids_test).sum() == 0, "Overlap detected: train ∩ test"
assert ids_val.isin(ids_test).sum() == 0, "Overlap detected: validation ∩ test"

print("Split integrity check passed: no customer overlap and row counts match.")

Split integrity check passed: no customer overlap and row counts match.


## 6. Feature Scaling

Question: Scaler phải được fit ở đâu để tránh data leakage?

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = pd.DataFrame(
    scaler.transform(X_train),
    columns=feature_cols,
).reset_index(drop=True)

X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=feature_cols,
).reset_index(drop=True)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=feature_cols,
).reset_index(drop=True)

print("Scaling completed: scaler fitted on train only, then applied to validation and test.")
print(f"X_train_scaled: {X_train_scaled.shape}")
print(f"X_val_scaled:   {X_val_scaled.shape}")
print(f"X_test_scaled:  {X_test_scaled.shape}")

Scaling completed: scaler fitted on train only, then applied to validation and test.
X_train_scaled: (29660, 95)
X_val_scaled:   (6356, 95)
X_test_scaled:  (6356, 95)


## 7. Shared Variables for Model Training

Các biến chung mà cả 3 thành viên sẽ dùng để train model:

| Biến | Mô tả |
|---|---|
| `X_train_scaled`, `X_val_scaled`, `X_test_scaled` | Feature đã chuẩn hóa |
| `y_train`, `y_val`, `y_test` | Nhãn (0 = Normal, 1 = Theft) |
| `ids_train`, `ids_val`, `ids_test` | CONS_NO, chỉ dùng tracking |

**Lưu ý:**
- `CONS_NO` chỉ dùng để tracking khách hàng, **không** dùng làm feature cho model.
- **Test set chỉ dùng cho final evaluation**, không dùng để chọn model hay tuning threshold.
- Dùng validation set để so sánh model và chọn threshold.